# Week 12C: A full-stack town Census tract explorer

Build a local web application that prepares **real Census ACS data for different towns** and maps Census variables at the **census tract level**. Choose a state and town, prepare its data, then switch towns and variables in an interactive choropleth.

[Download this notebook](lecture-12C.ipynb) · [Download the complete runnable app](downloads/census-full-stack.zip) · [App README](examples/census-full-stack/README.md) · [Previous: publishing](lecture-12B.html)

**Deliverable:** compare tract-level population, median household income, poverty rate, and median home value in Harrisburg, Pittsburgh, and a town of your choice. This version replaces the historical district/synthetic-dot radius demo. The older dot and district files remain available for Part A; Part C downloads its own tract data.
## Quick start: run the town tract explorer

Open a terminal in the folder containing this notebook, then run:

```bash
conda activate geospatial
conda install -c conda-forge fastapi uvicorn httpx shapely
cd examples/census-full-stack
python -m uvicorn app:app --host 127.0.0.1 --port 8000
```

For the standalone ZIP, use `cd census-full-stack` instead. Leave the terminal running and open **[http://127.0.0.1:8000/](http://127.0.0.1:8000/)**.

1. Select **Pennsylvania**, search **Harrisburg**, and choose **Harrisburg city — Incorporated place**.
2. Click **Prepare town data** and wait for the map.
3. Select a Census variable; click a tract to inspect its estimates and margin of error.
4. Search and prepare **Pittsburgh city — Incorporated place**, then compare the two towns using **Prepared town**.

The checked 2024 boundary selection produces **14 overlapping tracts for Harrisburg** and **128 for Pittsburgh**. Local installations that already have these towns prepared can select them immediately. A fresh download starts with no prepared towns.

**Expected map:** OpenStreetMap streets, colored census tracts, a purple dashed town boundary, and a numeric legend. Selecting a tract adds an orange boundary and a details popup. The table below the map shows all four Census measures. Whole-tract estimates may include areas outside the purple town boundary.

If you were running an older version, stop it with **Ctrl+C**, start the updated server, and hard-refresh the browser. This lecture HTML explains the app; the interactive app runs at the localhost URL above.


## 1. Understand the full stack

| Layer | Technology | Responsibility |
| --- | --- | --- |
| Frontend | HTML, CSS, JavaScript, Leaflet | Find a town, request preparation, choose a variable, show tract polygons and a table |
| Backend | Python, FastAPI, Uvicorn, Shapely | Download public data, select overlapping tracts, join estimates by GEOID, return GeoJSON |
| Local database | SQLite, included with Python | Save prepared towns and tract estimates between restarts |
| Public sources | Census TIGERweb and Census Reporter | Matching 2024 boundaries and 2020–2024 ACS five-year estimates |

The same Python process serves the web page and API at **http://127.0.0.1:8000/**. No frontend build step or separate CORS configuration is needed.

Part A teaches DuckDB Spatial queries on historical sample dots. This app uses Shapely for one-time boundary selection and SQLite for prepared-town storage. SQLite needs no Spatial extension download. You can inspect the SQL in `database.py`; no old Part A database is overwritten.

```text
Browser: choose state + town → POST /api/prepare
Python: town boundary → overlapping tract boundaries → ACS tables → join by tract GEOID
SQLite: save town and tract estimates
Browser: GET /api/tracts?town_id=...&variable=income → Leaflet choropleth + tract table
```

## 2. Use the project beside this notebook

In this course repository the app is under `examples/census-full-stack`. The existing `data` folder beside this notebook will hold the new downloads:

```text
week12-database-web-hosting/
├── lecture-12C.ipynb
├── data/
│   ├── census_sample.parquet        # Part A; not used by this app
│   ├── districts_2015.geojson       # Part A; not used by this app
│   └── tract-explorer/              # created by town preparation
│       ├── source-cache/           # original public JSON responses
│       └── towns/28-4232800/prepared.json
└── examples/census-full-stack/
    ├── app.py
    ├── database.py
    ├── prepare_data.py
    ├── test_app.py
    ├── static/
    └── data/tracts.sqlite           # prepared database; created automatically
```

Open this notebook with its lecture folder as the working directory. The following cell checks the app location without downloading anything.

**Standalone ZIP:** extract `census-full-stack` beside the notebook. In that layout, use `cd census-full-stack` instead of `cd examples/census-full-stack`. New source data still go into the sibling `data/tract-explorer` folder. Both layouts are detected automatically.

In [ ]:
from pathlib import Path

lecture_dir = Path.cwd()
app_dir = lecture_dir / "examples" / "census-full-stack"
if not app_dir.is_dir():
    app_dir = lecture_dir / "census-full-stack"  # standalone ZIP layout
assert (app_dir / "app.py").is_file(), "Open this notebook from its lecture folder."
print("Application:", app_dir.resolve())
print("Downloaded tract data:", (lecture_dir / "data" / "tract-explorer").resolve())

## 3. Install the app tools

Keep using the course **`geospatial`** environment. In a terminal:

```bash
conda activate geospatial
conda install -c conda-forge fastapi uvicorn httpx shapely
```

FastAPI defines routes and validates requests. Uvicorn runs the server. HTTPX downloads public JSON and supports the test client. Shapely selects tracts overlapping a town. SQLite comes with Python. The included `environment.yml` remains a dependency reference for the broader course environment.

For notebook cells, select **Python (geospatial)** or your existing kernel backed by that environment; restart it after installing packages.

**No API key is needed.** Estimates are from the public Census Reporter mirror of Census ACS, with boundaries directly from Census TIGERweb. Direct `api.census.gov` requests may require a Census API key; this app uses the mirror so classroom preparation works without one. The map uses free OpenStreetMap tiles and Leaflet. Initial preparation, the map library, and basemap tiles require internet access.

## 4. Start the application

In **Terminal 1**, start from the folder containing this notebook:

```bash
conda activate geospatial
cd examples/census-full-stack
python -m uvicorn app:app --host 127.0.0.1 --port 8000
```

For a standalone ZIP, replace the `cd` line with `cd census-full-stack`. Keep the terminal running.

Open:

- [Town Census tract explorer](http://127.0.0.1:8000/)
- [Interactive API documentation](http://127.0.0.1:8000/docs)
- [Health check](http://127.0.0.1:8000/api/health)

The database is created automatically. **You do not need to run the old `prepare_data.py --data ../../data` command.** Preparation now happens town by town in the browser.

In **Terminal 2**, verify the server:

```bash
curl http://127.0.0.1:8000/api/health
```

Expect `status: "ok"`, `year: 2024`, and a `prepared_towns` count, initially zero in a fresh installation. Existing prepared towns appear automatically after restarting the server.

Run the server in a terminal, not a notebook cell: the long-running process would block Run All. Use the Uvicorn URL; opening the HTML file, Live Server, or `python -m http.server` does not run the API. Press **Ctrl+C** to stop Uvicorn. After app edits, restart Uvicorn and hard-refresh the browser (Mac: Cmd+Shift+R; Windows/Linux: Ctrl+Shift+R).

## 5. Prepare Census data for different towns

### First example: Harrisburg, Pennsylvania

1. Select **Pennsylvania** and enter **Harrisburg**.
2. Click **Find towns**.
3. Choose **Harrisburg city — Incorporated place (`28:4232800`)**. A county-subdivision version may also appear; check the geography type.
4. Click **Prepare town data**. The first download can take a few minutes.
5. The app saves the data, selects the town, and displays the tract choropleth and town boundary.

### Second example: Pittsburgh, Pennsylvania

Search **Pittsburgh**, select **Pittsburgh city — Incorporated place (`28:4261000`)**, and prepare it. Both towns now appear in **Prepared town**. Switching between already prepared towns uses the local database.

### Your own town

Choose a state and search for a municipality. The search includes incorporated places, Census-designated places, and county subdivisions such as townships. Names can repeat; use the state, geography type, and Census identifier to select the intended area. The tool uses 2024 Census geography, so newly created or renamed municipalities may differ from today's names.

The backend fetches the selected town's boundary, finds overlapping tracts, downloads four ACS tables, and joins them by the **11-digit tract GEOID**. Repreparing replaces that town atomically and reuses cached public responses; other prepared towns remain saved. The source vintage is fixed at 2024 for reproducible teaching.

Prepared source responses and `prepared.json` packages are saved in `data/tract-explorer` beside the notebook; the database is `examples/census-full-stack/data/tracts.sqlite`. The old source files are not inputs to this tract explorer.

## 6. Read the choropleth and tract details

Choose **Median household income**, **Total population**, **Poverty rate**, or **Median owner-occupied home value**. The map and table update for the selected town.

| Variable | ACS table / estimate | Interpretation |
| --- | --- | --- |
| Total population | B01003, total | Estimated residents of the whole tract; count, not density |
| Median household income | B19013, median | 2024 inflation-adjusted dollars; not additive across tracts |
| Poverty rate | B17001, below poverty / poverty-status universe × 100 | Percent of people for whom poverty status is determined |
| Median owner-occupied home value | B25077, median | 2024 dollars for owner-occupied units; not all housing |

- **Purple dashed outline:** the selected town's boundary, with a white halo for visibility.
- **Tract fills:** the selected variable, using fixed class breaks for comparisons across towns.
- **Orange outline:** the tract you selected. Click a tract or its GEOID in the table to inspect it.
- **Gray fill:** missing or unavailable estimates; zero is a real value and remains distinct from missing.

Tract popups show all four measures and available **90% margins of error (MOEs)** for direct estimates. Poverty displays its numerator and denominator; this demo does not calculate a rate MOE. Large MOEs mean estimates are uncertain. `+` identifies top-coded medians (income $250,000+; home value $2,000,000+).

**Geography matters:** Census tracts and town boundaries do not necessarily nest. We select whole tracts with positive-area overlap; touching neighbors are excluded. The map retains complete tract boundaries and complete tract estimates, including areas outside the town. It does not allocate a fraction of a tract's population to the town. Do not add displayed tract values and call them a town total. Dollar medians are not sums or averages of household records.

All estimates describe the **2020–2024 ACS five-year period**. They are survey estimates, not current counts, a single-year census, or the old synthetic 2010 dots. Geometry is simplified slightly for display; selection uses the downloaded boundaries before simplification.

## 7. Optional command-line preparation

The browser is the simplest workflow. For a reproducible terminal workflow, run these commands from the app directory in Terminal 2:

```bash
conda activate geospatial
python prepare_data.py --state 42 --search Harrisburg
python prepare_data.py --town 28:4232800
python prepare_data.py --town 28:4261000
```

Search returns town IDs and geography types. Use those IDs rather than guessing FIPS codes. Reload the app to see towns prepared through the CLI. The server and preparation script use the same default database and source cache.

Optional overrides: `CENSUS_DB` changes the SQLite path, and `CENSUS_DATA_DIR` changes the source-cache/package folder. If overriding them, set the same values in both terminals. Do not point `CENSUS_DB` at the old DuckDB database; SQLite uses a different file format.

## 8. Inspect the API contract

| Request | Result |
| --- | --- |
| `GET /api/health` | Server status, vintage, number of prepared towns |
| `GET /api/config` | Variables and source descriptions |
| `GET /api/places?state=42&q=Harrisburg` | Matching towns and IDs |
| `POST /api/prepare` with `{"town_id":"28:4232800"}` | Download, join, and save the selected town |
| `GET /api/towns` | Locally prepared towns and tract counts |
| `GET /api/tracts?town_id=28:4232800&variable=income` | Whole-tract GeoJSON, chosen values, all metrics, town boundary, and provenance |

Use browser developer tools → **Network**. Observe the search, preparation POST, and choropleth GET. In `/docs`, try the tract endpoint after preparing a town. Supported variable keys are `population`, `income`, `poverty`, and `home_value`.

The next cell is an **optional live HTTP client**. Start Uvicorn and prepare Harrisburg first. This request talks to the Python server; it does not open the database directly.

In [ ]:
import httpx

response = httpx.get(
    "http://127.0.0.1:8000/api/tracts",
    params={"town_id": "28:4232800", "variable": "income"},
    timeout=30,
)
response.raise_for_status()
tracts = response.json()
print(tracts["town_name"], "—", len(tracts["features"]), "overlapping tracts")
[
    {"geoid": f["properties"]["geoid"], "income": f["properties"]["value"]}
    for f in tracts["features"][:5]
]

## 9. Trace the Python preparation and backend

Open [prepare_data.py](examples/census-full-stack/prepare_data.py). It uses the matching **ACS 2024 TIGERweb service** for both town and tract boundaries. Layers distinguish incorporated places, CDPs, county subdivisions, and tracts. Queries are paginated, and public responses are cached.

Shapely first finds candidate tracts in the town's bounding box, then retains those whose intersection with the town has positive area. Whole-tract geometries and estimates are preserved. Census Reporter requests explicitly use `acs2024_5yr`; the script checks the returned release and requires every selected tract to be present before replacing the saved town.

Open [database.py](examples/census-full-stack/database.py). The `towns` table stores the boundary and provenance; the `tracts` table stores one row per town and tract, including geometry and metrics. A transaction replaces one town's data without removing others.

Open [app.py](examples/census-full-stack/app.py). FastAPI validates query parameters and returns JSON/GeoJSON. Reads use bound SQL parameters:

```sql
SELECT * FROM tracts WHERE town_id = ? ORDER BY geoid
```

The browser passes a town ID, never SQL. Download errors leave previous database results available. Census sentinel codes become null rather than false zeros; a zero poverty denominator produces no rate. Town IDs encode a fixed, allowed TIGERweb layer and Census identifier, not an arbitrary remote URL.

## 10. Trace the frontend and test the system

Open [static/app.js](examples/census-full-stack/static/app.js). Search uses `fetch()` to obtain town matches; **Prepare town data** sends JSON in a POST. `loadTown()` fetches tract GeoJSON and draws fills with `L.geoJSON()`. A separate map pane draws the purple town boundary above the choropleth. Fixed color breaks allow comparison between towns; switching variables changes the legend and units.

Every tract has a tooltip, popup, and table row. Selecting a row zooms to that tract and highlights its boundary. A request counter discards outdated map responses during fast selections. The map reports unavailable libraries, tiles, API failures, and missing estimates.

From the app directory, run:

```bash
python -m unittest -v
```

The tests use **synthetic fixtures clearly isolated from the map data**, a temporary SQLite database, and mocked downloads. They require no prepared town, internet access, or running Uvicorn. They cover overlap versus touching, GEOID joins, missing values versus zeros, poverty denominators, top-coding, release mismatch, incomplete-download protection, repreparation, validation, and frontend routes.

Then test the real browser workflow:

1. Start with no prepared towns (fresh standalone install) and verify instructions appear.
2. Search and prepare Harrisburg; confirm tract fills and the purple town boundary appear.
3. Switch all four variables; check units, missing values, legend, and popup details.
4. Prepare Pittsburgh and switch between towns; the viewport fits the chosen town's tracts.
5. Click a tract and a table row; verify the orange selected-tract boundary.
6. Restart the server; verify prepared towns remain available.
7. Inspect `/api/tracts` and compare one popup against the saved `prepared.json`.

## 11. Troubleshooting and hosting

| Symptom | Check |
| --- | --- |
| Import error for FastAPI, HTTPX, or Shapely | Activate `geospatial` and install the section 3 packages |
| No prepared towns | Find a town and click **Prepare town data**; database creation alone downloads no town |
| No town matches | Confirm state and spelling; choose the intended incorporated-place, CDP, or township geography |
| Preparation fails | Check internet access to `tigerweb.geo.census.gov` and `api.censusreporter.org`; read the server terminal |
| Blank map / map library failed | Allow `unpkg.com`; hard-refresh the page after script updates |
| Tracts appear but streets do not | Check `tile.openstreetmap.org` requests; streets require internet access |
| Gray tract | Estimate unavailable or suppressed; inspect its popup and raw response rather than assuming zero |
| Old district/radius controls still appear | Stop the old server, start Uvicorn from this updated app directory, and hard-refresh |
| Port 8000 is occupied | Stop the previous server or choose `--port 8001`; use that URL for the browser and notebook client |
| File is not a database | Remove a custom `CENSUS_DB` override pointing at an old DuckDB file; use the default `data/tracts.sqlite` |

Stop Uvicorn before manually removing database files. Source-cache responses are fixed-vintage downloads. To refresh from the sources, stop the server, remove only `data/tract-explorer/source-cache` beside the notebook, restart, and reprepare the desired town. Repreparation retains other towns.

Quarto and GitHub Pages can publish this lecture but cannot run FastAPI. For deployment use a Python-capable host, persistent storage for the SQLite database and downloaded packages, a single Uvicorn worker for this teaching demo, and the host's HTTPS endpoint. Local preparation does not publish anything. Public multi-user deployment needs separate decisions about authentication, preparation jobs, and resource limits.

Render the updated notes from the lecture folder:

```bash
quarto render lecture-12C.ipynb --to html --no-execute
```

Keep the live server separate from notebook rendering. The optional HTTP cell requires it to be running.

## Exercises and sources

1. Compare median income in two towns using the same legend. Explain why a dark tract is not necessarily a dense tract.
2. Locate a tract crossing the town outline. Explain why its estimate cannot be treated as a town-only estimate.
3. Recalculate one tract's poverty rate from its displayed numerator and denominator.
4. Compare a direct estimate and its MOE; describe how uncertainty changes your interpretation.
5. Prepare a third town and explain which Census geography you selected and why.

**Primary dataset and boundary documentation:** [2024 ACS five-year geography](https://api.census.gov/data/2024/acs/acs5/geography.html), [median income variable](https://api.census.gov/data/2024/acs/acs5/variables/B19013_001E.json), [poverty numerator](https://api.census.gov/data/2024/acs/acs5/variables/B17001_002E.json), [Census TIGERweb ACS 2024 service](https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_ACS2024/MapServer).

**Estimate delivery:** [Census Reporter API documentation](https://github.com/censusreporter/census-api/blob/master/API.md). Census Reporter redistributes Census ACS estimates; the local prepared package records the requested and returned release and both source URLs.

**Map attribution:** [OpenStreetMap contributors](https://www.openstreetmap.org/copyright). Keep attribution visible when sharing map screenshots.